<a href="https://colab.research.google.com/github/ArlLps/Facom_LLMs/blob/main/Aula02_tokenizacao_pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 - Tokenização

## Parte 1 - Pré-tokenização


In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers

tok_ws = Tokenizer(models.BPE())
tok_ws.pre_tokenizer = pre_tokenizers.Whitespace()
frase = "Não, será punido o criminoso."

print(tok_ws.pre_tokenizer.pre_tokenize_str(frase))


## *Punctuation + Whitespace*

In [ ]:
tok_punc = Tokenizer(models.BPE())
tok_punc.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Punctuation()
])
print(tok_punc.pre_tokenizer.pre_tokenize_str(frase))

**Pretokenizer: ByteLevel - estilo GPT-2**

In [ ]:
tok_byte = Tokenizer(models.BPE())
tok_byte.pre_tokenizer = pre_tokenizers.ByteLevel()
print(tok_byte.pre_tokenizer.pre_tokenize_str(frase))

# Metaspace (SentencePiece style)

In [ ]:
tok_meta = Tokenizer(models.BPE())
tok_meta.pre_tokenizer = pre_tokenizers.Metaspace()
print(tok_meta.pre_tokenizer.pre_tokenize_str(frase))

#Treinamento


In [ ]:
# --------------------------------------------------------------------
# TREINAMENTO
# --------------------------------------------------------------------
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
import json

# 1. Configuração do Modelo (BPE)
tokenizer = Tokenizer(models.BPE())

# 2. Pré-tokenização ByteLevel
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# 3. Decodificador
tokenizer.decoder = decoders.ByteLevel()

# 4. Configuração do Treinador
trainer = trainers.BpeTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>"]
)

# 5. Carregar dados e Treinar
corpus = []
try:
    with open('corpus.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            corpus.append(data['text'])

    print(f"Treinando com {len(corpus)} frases...")
    tokenizer.train_from_iterator(corpus, trainer=trainer)

    # 6. Pós-processamento
    bos_id = tokenizer.token_to_id("<s>")
    eos_id = tokenizer.token_to_id("</s>")

    tokenizer.post_processor = processors.TemplateProcessing(
        single="<s> $A </s>",
        pair="<s> $A </s> </s> $B </s>",
        special_tokens=[("<s>", bos_id), ("</s>", eos_id)],
    )

    # 7. Salvar o JSON pronto para uso
    tokenizer.save("tokenizer_final_corrigido.json")
    print(f"Sucesso! Vocabulário: {tokenizer.get_vocab_size()}")

except FileNotFoundError:
    print("ERRO: O arquivo 'corpus_poetico_v3_expandido.jsonl' não foi encontrado. Gere o dataset primeiro!")

# Encode

In [ ]:
# 03_tokenizer_encode.ipynb
# Pipeline de tokenização: normalização → pré-tokenização → modelo → pós-processamento

from tokenizers import Tokenizer, normalizers, pre_tokenizers, processors
from tokenizers.normalizers import NFD, StripAccents, Lowercase
from tokenizers.pre_tokenizers import Whitespace, Digits, Sequence
from tokenizers.processors import TemplateProcessing

print("### Pipeline de tokenização ###")
print(" Normalization")
print(" Pre-tokenization")
print(" Model")
print(" Post-processing\n")

# Carregar o tokenizador treinado (BPE)
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

# -----------------------------------------------------------
# Normalization
# -----------------------------------------------------------
print("# Normalization")
normalizer = normalizers.Sequence([
    NFD(),          # decomposição de acentos
    Lowercase(),    # tudo minúsculo
    StripAccents()  # remove acentos
])
texto = "Héllò hôw are ü?"
print("Antes:", texto)
print("Depois:", normalizer.normalize_str(texto), "\n")
tokenizer.normalizer = normalizer

# -----------------------------------------------------------
# Pre-tokenization
# -----------------------------------------------------------
print("# Pre-tokenization")
pre_tok = Sequence([
    Whitespace(),
    Digits(individual_digits=True)
])
texto2 = "Hello! How are you? Tenho R$ 213,12."
print("Pré-tokenização:", pre_tok.pre_tokenize_str(texto2), "\n")
tokenizer.pre_tokenizer = pre_tok

# -----------------------------------------------------------
# Model
# -----------------------------------------------------------
print("# Model: BPE (Byte Pair Encoding)")
# já carregado do arquivo bpe_tokenizer.json

# -----------------------------------------------------------
# Post-processing
# -----------------------------------------------------------
from tokenizers.processors import TemplateProcessing

print("# Post-processing (TemplateProcessing) - Estilo RoBERTa")

# Precisamos pegar os IDs corretos que foram gerados no treino
bos_id = tokenizer.token_to_id("<s>") # Beginning of Sentence
eos_id = tokenizer.token_to_id("</s>") # End of Sentence

# Se por acaso retornou None (não achou), usamos valores padrão ou verificamos o vocab
if bos_id is None: bos_id = 1
if eos_id is None: eos_id = 2

tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> </s> $B </s>",
    special_tokens=[
        ("<s>", bos_id),
        ("</s>", eos_id),
    ],
)

print(f"Post-processor configurado com BOS: {bos_id} e EOS: {eos_id}")

# -----------------------------------------------------------
# Teste Final
# -----------------------------------------------------------
encoded = tokenizer.encode("olá mundo")
print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)

# -----------------------------------------------------------
# Aplicando tudo
# -----------------------------------------------------------
encoded = tokenizer.encode("olá mundo")
print("Tokens IDs:", encoded.ids)
print("Tokens:", encoded.tokens)


## Bytelevel vs SentencePiece

In [ ]:
# 03_bytelevel_vs_sentencepiece.ipynb
# Comparando ByteLevel (GPT-2) vs SentencePiece (mT5)

from transformers import AutoTokenizer
import unicodedata

# -----------------------------
# 1️⃣ Modelos
# -----------------------------
BYTELEVEL_MODEL = "openai-community/gpt2"
SENTPIECE_MODEL = "google/mt5-small"

tok_byte = AutoTokenizer.from_pretrained(BYTELEVEL_MODEL)
tok_spm  = AutoTokenizer.from_pretrained(SENTPIECE_MODEL)

# Garantir pad_token
if tok_byte.pad_token is None and hasattr(tok_byte, "eos_token"):
    tok_byte.pad_token = tok_byte.eos_token

# -----------------------------
# 2️⃣ Texto de exemplo
# -----------------------------
text = "Vamos comer, vovó! 🙂"
print(f"Texto: {text}\n")

# -----------------------------
# 3️⃣ Tokenização
# -----------------------------
def encode_details(tokenizer, name):
    enc = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]
    print(f"=== {name} ===")
    print("Tokens:", tokens)
    print("IDs:", ids)
    print("Qtd tokens:", len(tokens))
    print("Decoded:", tokenizer.decode(ids))
    print("Offsets:", offsets)
    print()

encode_details(tok_byte, "ByteLevel (GPT-2)")
encode_details(tok_spm, "SentencePiece (mT5)")

# -----------------------------
# 4️⃣ Comparação Unicode (opcional)
# -----------------------------
def show_unicode_chars(s):
    for ch in s:
        name = unicodedata.name(ch, "UNKNOWN")
        print(f"{repr(ch)} -> {name}")

print("\nCaracteres Unicode do texto:")
show_unicode_chars(text)


# Avaliação

In [ ]:
from tokenizers import Tokenizer
import numpy as np

# Carrega o tokenizador treinado
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

# Corpus de teste (pode ser parte do seu corpus real)
test_texts = [
    "O rato roeu a roupa do rei de Roma.",
    "Aprender tokenização é divertido!",
    "GPT-2 e mT5 usam abordagens diferentes.",
    "Python é ótimo para NLP 😄",
]

# Funções auxiliares
def count_chars(text):
    return len(text)

def count_words(text):
    return len(text.split())

def evaluate_tokenizer(tokenizer, texts):
    stats = []
    for t in texts:
        enc = tokenizer.encode(t)
        stats.append({
            "text": t,
            "chars": count_chars(t),
            "words": count_words(t),
            "tokens": len(enc.tokens),
            "unk": enc.tokens.count("<unk>"),
            "decoded_ok": (tokenizer.decode(enc.ids) == t)
        })
    return stats

stats = evaluate_tokenizer(tokenizer, test_texts)

# Converter para métricas agregadas
import pandas as pd
df = pd.DataFrame(stats)

tpc = (df["tokens"] / df["chars"]).mean()
tpw = (df["tokens"] / df["words"]).mean()
unk_rate = (df["unk"].sum() / df["tokens"].sum()) * 100
decode_acc = (df["decoded_ok"].mean()) * 100

print("=== Métricas de eficiência ===")
print(f"Tokens por caractere (TPC): {tpc:.3f}")
print(f"Tokens por palavra (TPW): {tpw:.3f}")
print(f"Percentual de <unk>: {unk_rate:.2f}%")
print(f"Reversibilidade (decode == original): {decode_acc:.1f}%")
print(f"Tamanho médio da sequência: {df['tokens'].mean():.1f} tokens/frase")


# Teste Final

In [ ]:
# -----------------------------------------------------------
# Célula de Correção e Visualização de Tokens
# -----------------------------------------------------------
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast

# 1. Carregar e corrigir o tokenizer BPE original
tokenizer = Tokenizer.from_file("bpe_tokenizer_melhorado.json")

# Restaurando a configuração ByteLevel (CRUCIAL)
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# Configurando BOS/EOS
bos_id = tokenizer.token_to_id("<s>")
eos_id = tokenizer.token_to_id("</s>")
tokenizer.post_processor = processors.TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> </s> $B </s>",
    special_tokens=[("<s>", bos_id), ("</s>", eos_id)],
)

# Salvar o JSON corrigido
tokenizer.save("tokenizer_final_corrigido.json")

# 2. Carregar no Transformers para teste
print("--- Visualizando os Tokens Gerados ---")
novo_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer_final_corrigido.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>"
)

# 3. Teste Detalhado
frase = "O silêncio devora a alma, ele entristece meu espírito e corrói minhas utopias"
inputs = novo_tokenizer(frase, return_tensors="pt")
ids = inputs['input_ids'][0]

tokens_visuais = novo_tokenizer.convert_ids_to_tokens(ids)

print(f"Frase Original: '{frase}'")
print(f"\n1. IDs (Números que o modelo vê):")
print(ids.tolist())

print(f"\n2. Tokens (Como o texto foi quebrado):")
print(tokens_visuais)

print(f"\n3. Decodificado (Texto reconstruído):")
print(f"'{novo_tokenizer.decode(ids, skip_special_tokens=True)}'")